## House Prices (Data Preparation — Feature Engineering)

In [1]:
import pandas as pd
from IPython.display import display

from house_prices_mlops.pipeline import load_stage
from house_prices_mlops.data_tracker import DataTracker
from house_prices_mlops.preparation.feature_engineering import (
    create_temporal_features, normalize_remodel_age, create_aggregate_features,
    create_presence_features, create_quality_area_interaction,
)

df_before = load_stage("missing")
tracker = DataTracker.load(df_before)
display(tracker.profile)


2026-10-11 22:13:56.478 | INFO     | house_prices_mlops.config:<module>:11 - PROJ_ROOT path is: /home/jovyan/notebooks/house_prices_cookiecutter/house-prices-mlops


,var,data_type,variable_type,unique_count,missing_count,missing_pct,zero_count,zero_pct,min,max,mean,median,std,skewness,kurtosis
0,Id,int64,identifier,1168,0,0.000000,0.0,0.0,1.0,1460.0,730.904966,732.5,425.369088,0.000920,-1.215850
1,MSSubClass,string,nominal,15,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,MSZoning,object,nominal,5,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,LotFrontage,float64,continuous,107,217,18.578767,0.0,0.0,21.0,313.0,70.343849,70.0,24.897021,2.406787,19.454319
4,LotArea,int64,continuous,890,0,0.000000,0.0,0.0,1300.0,215245.0,10689.642123,9600.0,10759.366198,11.958088,186.444082
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,MoSold,int64,temporal,12,0,0.000000,0.0,0.0,1.0,12.0,6.356164,6.0,2.670707,0.241761,-0.371218
77,YrSold,int64,temporal,5,0,0.000000,0.0,0.0,2006.0,2010.0,2007.818493,2008.0,1.322639,0.082593,-1.187097
78,SaleType,object,nominal,9,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
79,SaleCondition,object,nominal,6,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Temporal Features

Estas variables expresan la antigüedad de la vivienda en el momento de la venta. Los años originales indican fechas; sus diferencias permiten comparar viviendas por el tiempo transcurrido desde su construcción o remodelación.

- **HouseAge = YrSold − YearBuilt:** representa la antigüedad de la vivienda al venderse. Permite investigar cómo se relaciona esa antigüedad con su precio.
- **RemodAge = YrSold − YearRemodAdd:** representa los años transcurridos desde la remodelación registrada. Puede distinguir una vivienda antigua actualizada recientemente de otra sin una actualización reciente. Cuando YearRemodAdd coincide con YearBuilt, esta variable coincide con HouseAge.
- **GarageAge = YrSold − GarageYrBlt:** representa la antigüedad del garaje. Si GarageYrBlt falta, GarageAge permanece faltante; no se inventa un año de construcción. La función conserva el caso original GarageYrBlt=0 → GarageAge=0, aunque Missing no crea ese valor.

La entrada ya incluye Structure y las reglas comunes de Missing. Las columnas originales se conservan y las diferencias no utilizan SalePrice.

El original limita **RemodAge** a cero cuando resulta negativo. Es una regla de representación: una remodelación posterior a la venta produce una inconsistencia que se registra antes de aplicar el límite. El recorte no corrige los años originales ni demuestra que su información sea correcta.

In [2]:
df_hp = create_temporal_features(df_before)


In [3]:
display(
    df_hp[
        [
            'YearBuilt',
            'YearRemodAdd',
            'GarageYrBlt',
            'YrSold',
            'HouseAge',
            'RemodAge',
            'GarageAge'
        ]
    ].describe()
)

,YearBuilt,YearRemodAdd,GarageYrBlt,YrSold,HouseAge,RemodAge,GarageAge
count,1168.000000,1168.000000,1104.000000,1168.000000,1168.00000,1168.000000,1104.000000
mean,1970.965753,1984.897260,1978.662138,2007.818493,36.85274,22.921233,29.154891
std,30.675495,20.733955,24.728938,1.322639,30.70311,20.714585,24.745459
min,1872.000000,1950.000000,1900.000000,2006.000000,0.00000,-1.000000,0.000000
25%,1953.000000,1966.000000,1961.750000,2007.000000,7.00000,4.000000,6.000000
50%,1972.000000,1994.000000,1980.000000,2008.000000,35.00000,14.000000,27.500000
75%,2001.000000,2004.000000,2002.000000,2009.000000,55.00000,42.000000,47.000000
max,2010.000000,2010.000000,2010.000000,2010.000000,136.00000,60.000000,107.000000


In [4]:
display(
    df_hp.loc[
        df_hp['RemodAge'] < 0,
        [
            'YearBuilt',
            'YearRemodAdd',
            'YrSold',
            'HouseAge',
            'RemodAge'
        ]
    ]
)

,YearBuilt,YearRemodAdd,YrSold,HouseAge,RemodAge
523,2007,2008,2007,0,-1


In [5]:
negative_remodel = df_hp.loc[df_hp["RemodAge"].lt(0), ["Id", "YearRemodAdd", "YrSold", "RemodAge"]]
if not negative_remodel.empty:
    tracker.add_finding(
        key="remodel_year_after_sale",
        var="YearRemodAdd | YrSold",
        stage="DP - Feature Engineering - Temporal",
        finding=(
            f"{len(negative_remodel)} registros de entrenamiento presentan YearRemodAdd posterior a YrSold "
            f"y producen RemodAge negativo. Id: {negative_remodel['Id'].tolist()}."
        ),
    )


In [6]:
df_hp = normalize_remodel_age(df_hp)
tracker.log(
    var="RemodAge", stage="DP - Feature Engineering - Temporal",
    action="create_feature", method="year_difference_clipped",
    before=f"{len(negative_remodel)} records with negative RemodAge",
    after="RemodAge minimum constrained to 0",
)
if not negative_remodel.empty:
    tracker.resolve_finding(
        key="remodel_year_after_sale",
        resolution="RemodAge se limitó a un mínimo de cero, conservando YearRemodAdd y YrSold originales.",
    )


In [7]:
tracker.log(
    var='HouseAge',
    stage='DP - Feature Engineering',
    action='create_feature',
    method='YrSold - YearBuilt',
    before='YrSold, YearBuilt',
    after='HouseAge'
)

tracker.log(
    var='RemodAge',
    stage='DP - Feature Engineering',
    action='create_feature',
    method='YrSold - YearRemodAdd',
    before='YrSold, YearRemodAdd',
    after='RemodAge'
)

tracker.log(
    var='GarageAge',
    stage='DP - Feature Engineering',
    action='create_feature',
    method='YrSold - GarageYrBlt',
    before='YrSold, GarageYrBlt',
    after='GarageAge'
)

5

### Aggregate Features

Estas variables resumen características distribuidas entre varias columnas. Permiten evaluar si una medida conjunta aporta una representación útil, conservando también sus componentes originales.

- **TotalSF = TotalBsmtSF + GrLivArea:** combina la superficie del sótano con la superficie habitable sobre el nivel del suelo. Resume el tamaño conjunto registrado; no supone que ambas superficies tengan el mismo valor económico.
- **TotalBathrooms:** suma baños completos y medios baños, tanto sobre el nivel del suelo como en el sótano. Cada medio baño aporta 0.5, siguiendo la fórmula del original. Es un índice de capacidad, no una conversión física ni una equivalencia de precio demostrada.
- **TotalPorchSF:** suma WoodDeckSF, OpenPorchSF, EnclosedPorch, 3SsnPorch y ScreenPorch. Resume la superficie de esas áreas de uso exterior o complementario. El nombre procede del original, aunque incluye WoodDeckSF y espacios con características distintas.

Las fórmulas son fijas y no aprenden estadísticas. Su utilidad predictiva debe comprobarse durante Modeling; pueden aportar información redundante respecto a las columnas originales.

In [8]:
df_hp = create_aggregate_features(df_hp)


In [10]:
display(
    df_hp[
        [
            'TotalSF',
            'TotalBathrooms',
            'TotalPorchSF'
        ]
    ].describe()
)

,TotalSF,TotalBathrooms,TotalPorchSF
count,1168.000000,1168.000000,1168.000000
mean,2589.172945,2.222175,186.584760
std,823.125867,0.779502,161.131777
min,334.000000,1.000000,0.000000
25%,2041.000000,2.000000,45.000000
50%,2500.000000,2.000000,168.000000
75%,3028.250000,2.500000,275.250000
max,11752.000000,6.000000,1027.000000


In [11]:
tracker.log(
    var='TotalSF',
    stage='DP - Feature Engineering',
    action='create_feature',
    method='TotalBsmtSF + GrLivArea',
    before='TotalBsmtSF, GrLivArea',
    after='TotalSF'
)

tracker.log(
    var='TotalBathrooms',
    stage='DP - Feature Engineering',
    action='create_feature',
    method='FullBath + 0.5*HalfBath + BsmtFullBath + 0.5*BsmtHalfBath',
    before='FullBath, HalfBath, BsmtFullBath, BsmtHalfBath',
    after='TotalBathrooms'
)

tracker.log(
    var='TotalPorchSF',
    stage='DP - Feature Engineering',
    action='create_feature',
    method='WoodDeckSF + OpenPorchSF + EnclosedPorch + 3SsnPorch + ScreenPorch',
    before='WoodDeckSF, OpenPorchSF, EnclosedPorch, 3SsnPorch, ScreenPorch',
    after='TotalPorchSF'
)

8

### Presence / Binary Features

Estas variables separan la existencia de una característica de su magnitud. Por ejemplo, HasGarage permite distinguir viviendas con y sin garaje, mientras GarageArea describe su tamaño.

- **HasGarage:** indica si GarageArea es mayor que cero.
- **HasBasement:** indica si TotalBsmtSF es mayor que cero.
- **HasPool:** indica si PoolArea es mayor que cero.
- **HasFireplace:** indica si Fireplaces es mayor que cero.
- **IsRemodeled:** indica si YearRemodAdd difiere de YearBuilt. Es un indicador basado en los años registrados; no describe la magnitud o calidad de la remodelación y no identifica cambios realizados durante el mismo año de construcción.

Se conserva la codificación del original: **1** cuando se cumple la condición y **0** en caso contrario. Las comparaciones presuponen que las columnas fuente contienen información conocida. Con valores numéricos faltantes, una comparación como NaN > 0 puede producir False; por eso un cero no debe interpretarse como ausencia confirmada sin revisar los datos de entrada.

No se aprenden umbrales: se utilizan condiciones fijas sobre cada vivienda. Su contribución se evaluará en los experimentos.

In [12]:
df_hp = create_presence_features(df_hp)


In [13]:
binary_features = [
    'HasGarage',
    'HasBasement',
    'HasPool',
    'HasFireplace',
    'IsRemodeled'
]

for col in binary_features:
    print(f'\n{col}')
    print(df_hp[col].value_counts())

for col in binary_features:
    print(
        col,
        sorted(df_hp[col].unique())
    )


HasGarage
HasGarage
1    1104
0      64
Name: count, dtype: int64

HasBasement
HasBasement
1    1140
0      28
Name: count, dtype: int64

HasPool
HasPool
0    1162
1       6
Name: count, dtype: int64

HasFireplace
HasFireplace
1    621
0    547
Name: count, dtype: int64

IsRemodeled
IsRemodeled
0    601
1    567
Name: count, dtype: int64
HasGarage [np.int64(0), np.int64(1)]
HasBasement [np.int64(0), np.int64(1)]
HasPool [np.int64(0), np.int64(1)]
HasFireplace [np.int64(0), np.int64(1)]
IsRemodeled [np.int64(0), np.int64(1)]


In [14]:
binary_methods = {
    'HasGarage': 'GarageArea > 0',
    'HasBasement': 'TotalBsmtSF > 0',
    'HasPool': 'PoolArea > 0',
    'HasFireplace': 'Fireplaces > 0',
    'IsRemodeled': 'YearBuilt != YearRemodAdd'
}

for var, method in binary_methods.items():

    tracker.log(
        var=var,
        stage='DP - Feature Engineering',
        action='create_binary_feature',
        method=method,
        before='source variable(s)',
        after=var
    )

### Interaction Features

**QualGrLivArea = OverallQual × GrLivArea** representa conjuntamente la calidad general y la superficie habitable. Se crea para investigar si la relación entre superficie y precio varía con la calidad de la vivienda: una misma superficie puede tener implicaciones diferentes según su calidad.

El producto ofrece esa combinación de forma explícita. Puede ser útil para modelos que no construyen interacciones automáticamente; otros modelos pueden capturar relaciones semejantes a partir de las variables originales.

La multiplicación conserva la propuesta del original y trata la escala ordinal de OverallQual como un factor numérico. No demuestra que cada incremento de calidad sea equivalente ni que el efecto real sea multiplicativo. La correlación mostrada a continuación es una descripción sobre entrenamiento; la utilidad de esta variable debe verificarse mediante validación cruzada, comparando experimentos con y sin ella.

Su cálculo utiliza únicamente OverallQual y GrLivArea; no utiliza SalePrice ni parámetros aprendidos.

In [15]:
df_hp = create_quality_area_interaction(df_hp)
display(df_hp[["OverallQual", "GrLivArea", "QualGrLivArea"]].describe())


,OverallQual,GrLivArea,QualGrLivArea
count,1168.000000,1168.000000,1168.000000
mean,6.121575,1527.401541,9767.015411
std,1.367619,524.432686,5171.705396
min,1.000000,334.000000,334.000000
25%,5.000000,1145.750000,5926.500000
50%,6.000000,1473.000000,8877.000000
75%,7.000000,1792.000000,12351.000000
max,10.000000,5642.000000,56420.000000


In [16]:
display(df_hp[["OverallQual", "GrLivArea", "QualGrLivArea", "SalePrice"]].corr()["SalePrice"].sort_values(ascending=False))


SalePrice        1.000000
QualGrLivArea    0.816810
OverallQual      0.785555
GrLivArea        0.695652
Name: SalePrice, dtype: float64

In [17]:
tracker.log(
    var='QualGrLivArea',
    stage='DP - Feature Engineering',
    action='create_interaction_feature',
    method='OverallQual * GrLivArea',
    before='OverallQual, GrLivArea',
    after='QualGrLivArea'
)

14

### Integrated Common State

In [18]:
df_common = load_stage("common")
pd.testing.assert_frame_equal(df_hp, df_common)
pd.testing.assert_frame_equal(df_common[df_before.columns], df_before)
pd.testing.assert_frame_equal(df_before, load_stage("missing"))
assert len(df_common) == len(df_before)

tracker.update_profile(df_common)
# Conceptual classifications of derived features require explicit metadata approval.
display(tracker.profile.loc[~tracker.profile["var"].isin(df_before.columns)])


,var,data_type,variable_type,unique_count,missing_count,missing_pct,zero_count,zero_pct,min,max,mean,median,std,skewness,kurtosis
81,HouseAge,int64,None,121,0,0.000000,51.0,4.366438,0.0,136.0,36.852740,35.0,30.703110,0.620361,-0.428001
82,RemodAge,int64,None,61,0,0.000000,100.0,8.561644,0.0,60.0,22.922089,14.0,20.713616,0.505954,-1.277365
83,GarageAge,float64,None,99,64,5.479452,69.0,5.907534,0.0,107.0,29.154891,27.5,24.745459,0.658431,-0.380666
84,TotalSF,int64,None,822,0,0.000000,0.0,0.000000,334.0,11752.0,2589.172945,2500.0,823.125867,1.979121,15.055585
85,TotalBathrooms,float64,None,10,0,0.000000,0.0,0.000000,1.0,6.0,2.222175,2.0,0.779502,0.315672,0.065340
86,TotalPorchSF,int64,None,395,0,0.000000,203.0,17.380137,0.0,1027.0,186.584760,168.0,161.131777,1.102421,1.744364
87,HasGarage,int64,None,2,0,0.000000,64.0,5.479452,0.0,1.0,0.945205,1.0,0.227676,-3.917573,13.370271
88,HasBasement,int64,None,2,0,0.000000,28.0,2.397260,0.0,1.0,0.976027,1.0,0.153029,-6.232060,36.901760
89,HasPool,int64,None,2,0,0.000000,1162.0,99.486301,0.0,1.0,0.005137,0.0,0.071519,13.862369,190.491446
90,HasFireplace,int64,None,2,0,0.000000,547.0,46.832192,0.0,1.0,0.531678,1.0,0.499209,-0.127131,-1.987244


### Save Findings

In [19]:
tracker.save()
